# 실습 4. DSPy Optimizer — 프롬프트를 데이터로 자동 튜닝

## 실습 목표

손으로 프롬프트를 고치는 대신 **metric(좋은 답 기준) + trainset(예시)** 를 주면,
`BootstrapFewShot`이 few-shot 예시를 자동 부트스트랩해 프로그램을 컴파일합니다(교안 5장).

- 컴파일 전/후 dev 정확도를 비교
- ⚠ Optimizer는 LLM을 다회 호출(비용·시간↑) — trainset/devset을 작게

> `MLAPI_*` 필요. (실습 3의 RAG를 노트북 자립을 위해 다시 정의합니다.)

## 1. RAG 재정의 + metric (교안 5.2)

In [ ]:
from _common import (SAMPLE_DOCS, MLAPI_BASE_URL, MLAPI_API_KEY, MLAPI_MODEL, EMB_MODEL, have_mlapi)
import dspy

_emb=None; _dvec=None
def _retrieve(q,k=3):
    global _emb,_dvec
    if _emb is None:
        from sentence_transformers import SentenceTransformer
        _emb=SentenceTransformer(EMB_MODEL); _dvec=_emb.encode([d["text"] for d in SAMPLE_DOCS], normalize_embeddings=True)
    return [SAMPLE_DOCS[i] for i in (_dvec @ _emb.encode([q],normalize_embeddings=True)[0]).argsort()[::-1][:k]]

class GenerateAnswer(dspy.Signature):
    """문맥에 근거해 한국어로 간결히 답한다."""
    context=dspy.InputField(); question=dspy.InputField(); answer=dspy.OutputField()

class RAG(dspy.Module):
    def __init__(self,k=3):
        super().__init__(); self.k=k; self.generate=dspy.ChainOfThought(GenerateAnswer)
    def forward(self,question):
        ctx="\n".join(f"({d['topic']}) {d['text']}" for d in _retrieve(question,self.k))
        return self.generate(context=ctx, question=question)

def get_lm():
    return dspy.LM(f"openai/{MLAPI_MODEL}", api_base=MLAPI_BASE_URL, api_key=MLAPI_API_KEY, temperature=0.0, max_tokens=600)

def metric(example, pred, trace=None):
    """정답 키워드가 '모두' 답변에 있으면 1.0 (엄격 메트릭)."""
    a = pred.answer or ""
    return float(all(kw in a for kw in example.answer_key))
print("준비 완료")

## 2. trainset / devset + 컴파일 전후 비교 (교안 5.3~5.4)

**Optimizer 효과가 '눈에 보이려면' 베이스라인에 헤드룸이 있어야 한다.** 그래서 **정확한 한국어
전문용어**(코사인·재현율·정밀도·RRF·충실성)를 요구하는 엄격 메트릭을 쓴다 — 모델은 기본
프롬프트(zero-shot)에선 종종 **영어로 의역**(예: '재현율'을 'recall'로)해 메트릭을 놓친다.
BootstrapFewShot 이 trainset에서 **정답 용어를 쓰는 예시(demos)** 를 부트스트랩해 끼우면, 모델이
그 정확한 용어를 따라 써서 dev 정확도가 오른다. (train 5 / dev 5, temperature=0)

In [ ]:
# 정답 = '정확한 한국어 전문용어'. 기본 프롬프트는 자주 영어로 의역해 틀린다(헤드룸).
TRAIN_QA = [("벡터 검색이 가까운 문서를 찾을 때 쓰는 유사도 척도 이름은?", ("코사인",)),
            ("검색 평가에서 관련 문서를 얼마나 빠뜨리지 않았는지 보는 지표를 한국어로?", ("재현율",)),
            ("검색 평가에서 가져온 것 중 맞는 비율을 보는 지표를 한국어로?", ("정밀도",)),
            ("BM25와 벡터 결과를 순위로 합치는 기법의 약자는?", ("RRF",)),
            ("생성이 문맥에 충실한지 보는 지표를 한국어로?", ("충실성",))]
DEV_QA = [("벡터 임베딩 검색의 대표 유사도 척도를 한 단어로?", ("코사인",)),
          ("재현율과 함께 검색 정확도를 보는 한국어 지표는?", ("정밀도",)),
          ("두 검색 결과를 순위 기반으로 통합하는 약자는?", ("RRF",)),
          ("관련 문서를 놓치지 않는 정도를 보는 한국어 지표는?", ("재현율",)),
          ("답변이 근거 문맥을 잘 지키는지 보는 한국어 지표는?", ("충실성",))]

def accuracy(prog, devset):
    return sum(metric(ex, prog(question=ex.question)) for ex in devset) / len(devset)

dspy.configure(lm=get_lm())
trainset = [dspy.Example(question=q, answer_key=a).with_inputs("question") for q,a in TRAIN_QA]
devset   = [dspy.Example(question=q, answer_key=a).with_inputs("question") for q,a in DEV_QA]

base_rag = RAG(k=3)
before = accuracy(base_rag, devset)
print(f"[before] 기본 프롬프트 RAG  → dev 정확도 {before:.2f}")
from dspy.teleprompt import BootstrapFewShot
compiled = BootstrapFewShot(metric=metric, max_bootstrapped_demos=4, max_labeled_demos=4).compile(RAG(k=3), trainset=trainset)
after = accuracy(compiled, devset)
print(f"[after ] BootstrapFewShot  → dev 정확도 {after:.2f}  (Δ{after-before:+.2f})")

## 3. Optimizer가 '프롬프트를 어떻게 바꿨나' — base vs compiled 프롬프트 비교

같은 질문에 대해 **컴파일 전(base)** 과 **컴파일 후(compiled)** 가 LLM에 보낸 프롬프트를 비교한다.
compiled 쪽 프롬프트엔 Optimizer가 **few-shot 예시(demos)를 끼워넣어** 메시지 수가 늘고, 모델이
그 예시의 정확한 용어를 따라 써 답이 바뀐다(예: 'recall' → '재현율').

In [ ]:
q = "관련 문서를 놓치지 않는 정도를 보는 한국어 지표는?"   # base가 'recall'로 틀리던 질문
ex_q = dspy.Example(question=q, answer_key=("재현율",))

bp = base_rag(question=q);  base_msgs = dspy.settings.lm.history[-1]["messages"]
cp = compiled(question=q);  comp_msgs = dspy.settings.lm.history[-1]["messages"]

print(f"● base     답변: {bp.answer.strip()[:40]!r}   통과={bool(metric(ex_q, bp))}")
print(f"● compiled 답변: {cp.answer.strip()[:40]!r}   통과={bool(metric(ex_q, cp))}")
print(f"\n[프롬프트 메시지 수]  base {len(base_msgs)}개  →  compiled {len(comp_msgs)}개")
print(f"  = Optimizer가 few-shot 예시(demos) {len(compiled.generate.predict.demos)}개를 프롬프트에 삽입 → 길어짐")

print("\n---- Optimizer가 삽입한 few-shot 예시(일부) — 정답 용어를 쓰는 시범 ----")
for i, demo in enumerate(compiled.generate.predict.demos[:2], 1):
    print(f"[demo {i}] Q: {getattr(demo, 'question', '')[:38]}")
    print(f"         A: {getattr(demo, 'answer', '')[:60]}")

**포인트 — Optimizer가 한 일 = '프롬프트에 정답 예시를 끼워 학습'**

- **before 0.6 → after 0.8~1.0** (실측, 실행마다 ±변동): 기본 프롬프트는 '재현율'을 'recall'로 의역해 틀리지만, 컴파일 후엔 demos가 정확 용어를 가르쳐 통과
- 프롬프트가 **2개 메시지 → 8개 안팎**으로: Optimizer가 (Q→A) few-shot 예시를 자동 삽입(우리가 손으로 쓴 게 아님)
- **효과가 보이는 조건**: 헤드룸 있는 엄격 메트릭 + 베이스라인이 틀리는 케이스. (강한 모델·쉬운 과제면 변화가 작다 — 그래서 '정확 용어' 과제로 설계)
- 더 작은 모델(OpenRouter gpt-oss-20b 등)을 쓰면 헤드룸이 더 커 개선폭↑ — 단 본 환경엔 OpenRouter 키가 없어 **같은 모델에서 과제 난이도로 헤드룸을 만든** 것
- 지시문까지 자동 탐색하는 **MIPRO**(`MIPROv2`)는 더 큰 개선 여지(5교시 PPTX 5.3 참고)

# [TODO]

## [TODO] 1. metric 함수 완성

`my_metric` 은 **정답 키워드(answer_key의 두 단어)가 모두 답변에 있으면 1.0, 아니면 0.0** 을
반환해야 합니다. 비어 있는 부분을 채우세요.

In [ ]:
def my_metric(example, pred, trace=None):
    a = pred.answer or ""
    # [TODO] 1: answer_key의 모든 키워드가 a에 있으면 1.0
    # [TODO] 여기에 코드를 작성하세요.
    pass

# 검증(LLM 없이)
class P: answer = "RAG는 검색으로 보강하고 파인튜닝은 가중치를 학습한다"
ex = dspy.Example(answer_key=("검색","가중치"))
print("metric:", my_metric(ex, P()))   # 1.0 기대

## [TODO] 2. base RAG의 dev 정확도만 측정

컴파일 없이 **기본 RAG(`RAG(k=3)`)** 의 dev 정확도만 `accuracy(...)`로 측정해 출력하세요.

In [ ]:
# [TODO] 2: base 정확도만 측정
# [TODO] 여기에 코드를 작성하세요.


## 실습 정리

- Optimizer = metric + trainset 으로 few-shot 예시를 **자동 탐색·삽입** → 프롬프트가 바뀐다(2→10 메시지)
- **헤드룸 있는 과제**(정확 용어 메트릭)에서 dev 0.6 → 0.8~1.0 으로 개선 — 'recall'→'재현율'을 demos가 가르침
- 컴파일은 오프라인 1회, 서빙은 컴파일된 프로그램 재사용
- 지시문까지 자동 최적화하려면 **MIPROv2**(5교시) — 더 큰 개선 여지·비용↑